# 10장 — 품질 성적표: 양자화는 얼마나 잃나 📉

웹 챕터의 무릎 곡선을 코드로 재현합니다. 미니 언어모델을 직접 양자화해 퍼플렉시티
열화를 재고, **열화 = KL 발산(5부)** 임을 수식 두 경로로 검산합니다.

> **노트북 사용법**: 각 셀을 클릭하고 `Shift + Enter`를 누르며 위에서 아래로 내려가세요.

**이 노트북의 목표**: 성적표를 스스로 만드는 힘.
**"무릎(스윗 스팟)을 찾고, 작은 차이는 오차범위를 의심한다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch10')

## 1. 미니 언어모델 준비: bigram 카운트로 로짓 만들기 (웹과 동일)

In [ ]:
import numpy as np

CORPUS = 'the cat sat on the mat and the rat ate the hat then the cat ran to the man '
chars = sorted(set(CORPUS))
V = len(chars)
idx = {c: i for i, c in enumerate(chars)}

counts = np.zeros((V, V))
for a, b in zip(CORPUS, CORPUS[1:]):
    counts[idx[a], idx[b]] += 1
logits = np.log(counts + 0.5)                       # 스무딩 — 안 본 조합도 눈금 위에
freq = counts.sum(axis=1); freq = freq / freq.sum() # 문맥 등장 빈도(가중치)

def softmax(row):
    m = row.max()                                    # 6장의 max 빼기 — 여기서도 현역
    e = np.exp(row - m)
    return e / e.sum()

P_fp = np.array([softmax(r) for r in logits])
print(f'어휘 {V}자, 로짓 행렬 {logits.shape}, 미니 모델 준비 완료')
assert V == 12
print('✔ 확인: 모델 = 숫자 상자(로짓 행렬). 이걸 양자화하면 무엇을 잃는지 잰다')

## 2. 무릎 곡선: b비트 양자화 후 퍼플렉시티

기준: **원본(fp) 모델의 분포**를 진실로 놓고 양자화 모델을 채점합니다. 열화가 음수가 될 수 없는
깨끗한 측정입니다.

In [ ]:
def quantize_logits(L, bits):
    levels = 2 ** (bits - 1) - 1
    s = np.abs(L).max(axis=1, keepdims=True) / levels
    return np.round(L / s) * s

def ppl_against_fp(L):
    Pq = np.array([softmax(r) for r in L])
    H = -(freq[:, None] * P_fp * np.log2(Pq)).sum()   # 가중 크로스 엔트로피 (비트)
    return 2 ** H

fp_ppl = ppl_against_fp(logits)
print(f'F16(원본)  PPL = {fp_ppl:.3f}')
curve = {}
for b in [8, 6, 4, 3, 2]:
    curve[b] = ppl_against_fp(quantize_logits(logits, b))
    print(f'int{b}      PPL = {curve[b]:.3f}   (열화 {(curve[b] / fp_ppl - 1) * 100:+.2f}%)')

assert abs(fp_ppl - 7.119) < 0.01, '웹 실험 ①과 동일 수치'
assert curve[4] / fp_ppl < 1.005, '16→4비트: 열화 +0.5% 미만 — 사실상 공짜 구간'
assert curve[2] / fp_ppl > 1.05, '2비트: +8% 급 폭등 — 무릎 아래 낭떠러지'
assert fp_ppl <= curve[8] + 1e-9 and curve[8] <= curve[4] + 1e-3 <= curve[3] + 1e-3 <= curve[2], '열화는 (사실상) 단조 — fp 아래로 내려가지 않는다'
assert all(curve[b] >= fp_ppl - 1e-9 for b in curve), '음수 열화 없음 — KL ≥ 0 의 보장'
print()
print('✔ 확인: 무릎이 4비트 근처. 국민 태그 q4 는 우연이 아니라 이 곡선 위에 앉아 있다')

## 3. 열화의 정체 검산: H(p, q) = H(p) + KL(p‖q)

5부 4장의 분해식을 **두 경로로 각각 계산해** 같은지 확인합니다. 열화 = 초과 비트(KL).

In [ ]:
Lq = quantize_logits(logits, 2)                      # 가장 극적인 int2 로 검산
P_q = np.array([softmax(r) for r in Lq])

H_fp = -(freq[:, None] * P_fp * np.log2(P_fp)).sum()          # 원본의 엔트로피 (5부 3장)
H_cross = -(freq[:, None] * P_fp * np.log2(P_q)).sum()        # 크로스 엔트로피 (5부 4장)
KL = (freq[:, None] * P_fp * np.log2(P_fp / P_q)).sum()       # KL 발산 (5부 5장)

print(f'H(p_fp)          = {H_fp:.4f} 비트')
print(f'H(p_fp, p_q)     = {H_cross:.4f} 비트')
print(f'KL(p_fp ‖ p_q)   = {KL:.4f} 비트  ← 틀린 지도의 초과 비트')

assert np.isclose(H_cross, H_fp + KL, atol=1e-9), '분해식 성립 — 열화분이 정확히 KL'
assert KL > 0, '깁스 부등식 — 진짜 분포보다 잘 맞히는 지도는 없다'
assert np.isclose(2 ** H_cross, curve[2], atol=1e-9), 'PPL = 2^크로스 엔트로피 (5부 7장)'
print()
print('✔ 확인: 5부 세 장을 한꺼번에. 크로스 엔트로피(4장) = 엔트로피(3장) + KL(5장), PPL(7장) = 2^H')

## 4. 무릎 곡선 그리기 + 실전 성적표 추천기

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

bits_axis = [16, 8, 6, 4, 3, 2]
ppls = [fp_ppl] + [curve[b] for b in [8, 6, 4, 3, 2]]
deg = [(p / fp_ppl - 1) * 100 for p in ppls]

plt.figure(figsize=(6.5, 3.8))
plt.plot(range(len(bits_axis)), deg, 'o-', color='tab:purple')
plt.xticks(range(len(bits_axis)), ['F16', '8bit', '6bit', '4bit', '3bit', '2bit'])
plt.axvline(3, color='tab:orange', ls='--', label='무릎(스윗 스팟): q4 가 앉은 자리')
plt.ylabel('퍼플렉시티 열화 (%)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('무릎 곡선: 4비트까지는 거의 공짜, 그 아래는 낭떠러지'); plt.tight_layout(); plt.show()

# 실전 성적표(교육용 수치)에서 RAM 별 최선 고르기 — 웹 실험 ②
REPORT = {'F16': (13.1, 5.80), 'Q8_0': (7.0, 5.80), 'Q6_K': (5.4, 5.81),
          'Q4_K_M': (4.0, 5.87), 'Q3_K_M': (3.2, 6.10), 'Q2_K': (2.1, 7.40)}

def recommend(ram_gb):
    """어림 규칙: 파일 + 4GB 여유(OS·KV 몫) ≤ RAM 중 PPL 최소."""
    fits = {t: (gb, ppl) for t, (gb, ppl) in REPORT.items() if gb <= ram_gb - 4}
    return min(fits, key=lambda t: fits[t][1])

for ram in [8, 16, 32]:
    print(f'RAM {ram:>2}GB → 추천 {recommend(ram)}')
assert recommend(8) == 'Q4_K_M' and recommend(16) == 'Q8_0' and recommend(32) == 'F16'
print()
print('✔ 확인: 성적표 + 어림 규칙만으로 내 PC 의 최선이 계산된다. 정밀 정산은 11장 캡스톤')

**직접 해보기**: ① CORPUS 에 문장을 덧붙여 fp PPL 이 어떻게 변하는지 보세요(자가 바뀌면
눈금도 바뀐다. 성적표의 전제). ② s2 를 코퍼스 텍스트로 다시 채점하도록 바꿔 보면
toy 에선 열화가 가끔 음수처럼 보이는 잔물결이 생깁니다. fp 기준 측정이 깨끗한 이유죠.

## 5. 관찰 정리 + 🤖 AI 연결

- **측정법**: 같은 기준으로 전/후 PPL 비교. fp 분포 기준이면 **열화 = KL(p_fp‖p_q) ≥ 0**
  (H(p,q) = H(p) + KL, 5부 4장 분해식, 두 경로 검산 완료).
- **무릎 곡선**: 16→4비트 +0.2%(공짜) → 3비트 +0.7% → 2비트 +8%. **q4 = 무릎 위의 국민 태그**.
  미시 원인: 눈금 부족으로 분포가 뭉개져 정답 확률이 깎임.
- **성적 읽는 눈**(6부 9장): 작은 차이는 오차범위 의심, PPL 은 도메인을 탐(다각 평가),
  전제는 같은 텍스트·같은 토크나이저. RAM 추천: 8GB → Q4_K_M / 16GB → Q8_0 / 32GB → F16.
- **다시 쓰는 수학**: 5부 3·4·5·7장(엔트로피·크로스 엔트로피·KL·퍼플렉시티, 네 개념을 한꺼번에!),
  6부 9장(오차범위), 6장(max 빼기).

**다음 챕터**: 🎯 11장 캡스톤. "7B는 왜 4GB인가"에 스스로 답하고, KV 캐시까지 정산해
내 PC에 맞는 모델을 고르는 졸업 시험.